# 05 - Explaining one model: MASTER, DJI, test year 2020

Step 5 of `THESIS_GUIDE.md` (PIANO.md B7). Before running the explanations on the whole grid, every method is looked at on one run: MASTER, DJI, T = 20, L = 5, seed 42, test year 2020.

**What is explained.** On every other weekly rebalance date of the test year (about 26 days, the days the top-k portfolio trades on), each stock's output is attributed to that stock's own inputs, a T x F window (20 days x 157 Alpha158 features + 63 market features). Two outputs (decision D4): the score (`pred`) and the **top-k margin** (`topk_margin`: the score minus the score the stock has to beat to be among the k = 5 held stocks), which explains portfolio membership rather than the score.

**Methods** (`xaifin/xai/`):
- **Integrated Gradients** (`gradient.py`): the path from a baseline to the input, 16 Gauss-Legendre steps. Baseline: the day's average stock (cross-sectional mean input), so an attribution says why this stock scores differently from the average stock of that day. Path "joint": the whole day moves from the baseline together and each stock keeps the gradient through its own inputs (decisions of 2026-11-04 in PIANO.md; section 6 checks them).
- **Saliency**: the gradient at the input.
- **Permutation importance by family** (`permutation.py`): the inputs of one family are shuffled across the stocks of the day; the drop in RankIC and the share of the top-k replaced measure what the family is worth for ranking and for the portfolio.
- **MASTER's own weights** (`intrinsic.py`): the feature gate, the temporal and the inter-stock attention, as supporting evidence only.

Importance is aggregated (`aggregate.py`) as the share of |attribution| each feature, lag or family gets, per stock-day, averaged.

In [1]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin.analysis.agreement import agreement, agreement_matrix
from xaifin.config import RESULTS_ROOT, TOP_K
from xaifin.models.registry import load_run
from xaifin.xai.aggregate import global_importance, input_table
from xaifin.xai.base import explain_dates, explain_run, load_attributions, save_attributions, xai_dir
from xaifin.xai.gradient import IntegratedGradients, Saliency, completeness_gap
from xaifin.xai.intrinsic import run_weights
from xaifin.xai.permutation import FamilyPermutation, summarize

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
RUN = RESULTS_ROOT / "Regression/MASTER/dji/sl20_pl5/seed42/y2020"
adapter = load_run(RUN)
K = TOP_K[adapter.cfg.universe]
dates = explain_dates(adapter)
inputs = input_table(adapter)
names = np.array(adapter.feature_names)
print(f"{adapter.cfg.model} {adapter.cfg.universe} T={adapter.cfg.seq_len} L={adapter.cfg.pred_len} seed {adapter.cfg.seed} "
      f"y{adapter.cfg.test_year}: {len(dates)} explained days, k = {K}, {len(inputs)} inputs per stock")

MASTER dji T=20 L=5 seed 42 y2020: 27 explained days, k = 5, 4400 inputs per stock


## 1. Attributions

Read from the results store (`scripts/explain.py` writes them); computed and saved here if the run is not explained yet.

In [2]:
methods = {"ig_joint": IntegratedGradients(), "saliency": Saliency()}
attr, glob = {}, {}
for name, explainer in methods.items():
    for target in ("pred", "topk_margin"):
        result = load_attributions(adapter, name, target)
        if result is None:
            result = explain_run(adapter, explainer, target, K, dates)
            out = save_attributions(adapter, name, target, result, explainer.params())
            global_importance(result["attr"], result["date"], inputs).to_parquet(out / f"global_{target}.parquet")
        attr[name, target] = result
        glob[name, target] = global_importance(result["attr"], result["date"], inputs)
print({key: r["attr"].shape for key, r in attr.items()})

{('ig_joint', 'pred'): (737, 20, 220), ('ig_joint', 'topk_margin'): (737, 20, 220), ('saliency', 'pred'): (737, 20, 220), ('saliency', 'topk_margin'): (737, 20, 220)}


## 2. One stock on one day

The IG attribution of the stock with the highest score on the first explained day, for the 30 inputs (feature x lag) with the largest attributions. Red: the input raises the score above the average stock's, blue: lowers it.

In [3]:
r = attr["ig_joint", "pred"]
day = r["date"][0]
rows = np.flatnonzero(r["date"] == day)
i = rows[np.argmax(r["score"][rows])]
a = r["attr"][i]
top = np.argsort(-np.abs(a).sum(0))[:30]
lags = [f"lag {a.shape[0] - 1 - t}" for t in range(a.shape[0])]
px.imshow(a[:, top], x=names[top], y=lags, color_continuous_scale="RdBu_r", color_continuous_midpoint=0, aspect="auto",
          title=f"IG, {r['ticker'][i]} on {pd.Timestamp(day).date()} (highest score of the day)").show()

## 3. Which inputs matter: features, families, lags; score against top-k membership

Share of the |IG attribution| per feature (summed over lags), per family and per lag, averaged over stock-days; error bars: standard deviation over days. `signed`: the same share with its sign.

In [4]:
g = {t: glob["ig_joint", t] for t in ("pred", "topk_margin")}
display(pd.concat({t: g[t][g[t]["level"] == "feature"].head(20)[["name", "importance", "signed"]].reset_index(drop=True) for t in g}, axis=1))
fam = pd.concat([g[t][g[t]["level"] == "family"].assign(target=t) for t in g])
px.bar(fam, x="name", y="importance", color="target", barmode="group", error_y="std_days",
       title="IG: share of the attribution per family").show()
lag = pd.concat([g[t][g[t]["level"] == "lag"].assign(target=t) for t in g])
lag["lag"] = lag["name"].astype(int)
px.line(lag.sort_values("lag"), x="lag", y="importance", color="target", markers=True,
        title="IG: share of the attribution per lag (0 = last input day)").show()
vec = lambda gl, level: gl[gl["level"] == level].set_index("name")["importance"]
pd.DataFrame({level: agreement(vec(g["pred"], level), vec(g["topk_margin"], level)) for level in ("feature", "family", "lag")}).T

pred                    topk_margin                   
       name importance  signed        name importance  signed
0    CNTP20     0.0146  0.0006      CNTP20     0.0148  0.0006
1    WVMA10     0.0118 -0.0000      WVMA10     0.0117 -0.0000
2    BETA20     0.0117 -0.0000      QTLD20     0.0117  0.0001
3    QTLD20     0.0117  0.0001      BETA20     0.0116 -0.0000
4    VSUMN5     0.0116 -0.0003      VSUMN5     0.0115 -0.0003
5    VSUMD5     0.0114 -0.0014      VSUMD5     0.0114 -0.0014
6    BETA30     0.0113 -0.0003      BETA30     0.0113 -0.0003
7   VSUMP30     0.0113 -0.0005     VSUMP30     0.0113 -0.0005
8     STD60     0.0110  0.0005       STD60     0.0109  0.0005
9     MAX60     0.0109 -0.0000       MAX60     0.0108 -0.0000
10   RESI20     0.0106 -0.0018      RESI20     0.0106 -0.0017
11    STD10     0.0105  0.0007       STD10     0.0105  0.0007
12   RESI30     0.0104 -0.0016      RESI30     0.0105 -0.0016
13   CORR60     0.0104  0.0002      CORR60     0.0104  0.0002
14   VSTD20     0.0103 -0.0002      VSTD20     0.0104 -0.0002
15     ROC5     0.0101  0.0001        ROC5     0.0100  0.0001
16    OPEN0     0.0100 -0.0009       OPEN0     0.0100 -0.0009
17  VSUMP20     0.0099  0.0006     VSUMP20     0.0100  0.0006
18   QTLU20     0.0099  0.0001      QTLU20     0.0098  0.0001
19     KSFT     0.0091  0.0002      RESI60     0.0091 -0.0004

,names,spearman,kendall,jaccard,rbo
feature,220.0000,0.9998,0.9918,1.0000,0.9703
family,8.0000,1.0000,1.0000,1.0000,1.0000
lag,20.0000,1.0000,1.0000,1.0000,1.0000


## 4. Saliency and permutation importance against IG

Family rankings of the three methods (the market family is left out: the day-mean baseline and the cross-sectional shuffle cannot see it, see section 5). Permutation importance: `d_RankIC`, the drop in RankIC when the family is shuffled across stocks, and `topk_change`, the share of the 5 held stocks that change.

In [5]:
perm_dir = xai_dir("permutation", adapter)
perm_rows = (pd.read_parquet(perm_dir / "permutation.parquet") if (perm_dir / "permutation.parquet").exists()
             else FamilyPermutation().explain_run(adapter, dates, K))
perm = summarize(perm_rows)
display(perm)
family = lambda gl: gl[(gl["level"] == "family") & (gl["name"] != "market")].set_index("name")["importance"]
vectors = {"IG": family(glob["ig_joint", "pred"]), "Saliency": family(glob["saliency", "pred"]),
           "IG top-k": family(glob["ig_joint", "topk_margin"]),
           "perm. top-k change": perm.set_index("family")["topk_change"], "perm. dRankIC": perm.set_index("family")["d_RankIC"]}
display(agreement_matrix(vectors, "spearman"))
feature = lambda gl: gl[gl["level"] == "feature"].set_index("name")["importance"]
agreement(feature(glob["ig_joint", "pred"]), feature(glob["saliency", "pred"]))

,family,d_RankIC,d_RankIC_std,topk_change,topk_change_std,score_change,score_change_std
0,trend,-0.0069,0.1988,0.5333,0.1403,0.7414,0.1651
1,position,0.0038,0.1206,0.4741,0.1157,0.6519,0.1648
2,volume,0.0291,0.1045,0.3926,0.1612,0.4712,0.1269
3,momentum,0.0306,0.1309,0.3644,0.1617,0.5311,0.2253
4,volatility,0.0281,0.1045,0.3526,0.1447,0.4780,0.1914
5,candle,-0.0033,0.0993,0.2800,0.1137,0.3290,0.0759
6,price_volume,-0.0091,0.0725,0.2459,0.1015,0.2615,0.0708


,IG,Saliency,IG top-k,perm. top-k change,perm. dRankIC
IG,1.0000,0.9643,1.0000,0.7500,0.6429
Saliency,0.9643,1.0000,0.9643,0.7143,0.5714
IG top-k,1.0000,0.9643,1.0000,0.7500,0.6429
perm. top-k change,0.7500,0.7143,0.7500,1.0000,0.2143
perm. dRankIC,0.6429,0.5714,0.6429,0.2143,1.0000


{'names': 220,
 'spearman': np.float64(0.8962713458351922),
 'kendall': np.float64(0.7206988526254482),
 'jaccard': 0.42857142857142855,
 'rbo': 0.39245183078693086}

## 5. MASTER's own weights

The **feature gate** reweights the 157 Alpha158 features from the market features of the last input day (the same weights for every stock, mean 1): it is the channel through which the market features act, invisible to IG with the day-mean baseline. **Temporal attention**: the weight of each day of the window. **Inter-stock attention**: how much a stock attends to the others; 1/N for every pair would be uniform.

In [6]:
if adapter.cfg.model == "MASTER":
    w = run_weights(adapter, dates)
    print("replay of the forward pass, largest difference from the model's output:", w["max_diff"])
    gate = pd.DataFrame(w["gate"], columns=names[:w["gate"].shape[1]], index=pd.to_datetime(w["date"]))
    gate_summary = gate.agg(["mean", "std"]).T.sort_values("mean", ascending=False)
    display(gate_summary.head(15))
    px.line(gate[gate_summary.index[:5]], title="Feature gate of the 5 most up-weighted features, per explained day").show()
    temporal = np.concatenate(w["temporal"]).mean(0)
    px.bar(x=[len(temporal) - 1 - t for t in range(len(temporal))], y=temporal, labels={"x": "lag", "y": "mean weight"},
           title="Temporal attention per lag (0 = last input day)").show()
    self_weight = [np.diag(m).mean() * len(m) for m in w["inter"]]
    print(f"inter-stock attention on oneself, relative to uniform (1 = uniform): {np.mean(self_weight):.2f}")
    ig_feat = feature(glob["ig_joint", "pred"])
    print("gate mean vs IG feature importance (Alpha158 features):", agreement(gate_summary["mean"], ig_feat))

replay of the forward pass, largest difference from the model's output: 0.0


,mean,std
KSFT2,1.3027,0.1812
CNTD30,1.2918,0.2978
CORR60,1.2585,0.2315
CORR5,1.2199,0.2073
IMAX5,1.1993,0.1992
KMID,1.1920,0.1974
RESI20,1.1843,0.1486
RESI60,1.1732,0.2439
QTLU60,1.1698,0.1907
WVMA10,1.1651,0.2310


inter-stock attention on oneself, relative to uniform (1 = uniform): 1.00
gate mean vs IG feature importance (Alpha158 features): {'names': 157, 'spearman': np.float64(0.3192093724223048), 'kendall': np.float64(0.2209701126898579), 'jaccard': 0.05263157894736842, 'rbo': 0.09331231744078049}


## 6. Checks of the IG settings

On the first three explained days: (a) the path, 'joint' (used) against the guide's 'own' (only the stock's inputs move, the other stocks stay; exact completeness per stock); (b) the baseline, the day's average stock (used) against zero (the training median); (c) completeness of 'own' (sum of the attributions minus the change of the target from the baseline), which also checks that 16 steps suffice.

In [7]:
checks = []
for batch in adapter.day_batches("test"):
    if batch.date not in dates[:3]:
        continue
    joint = IntegratedGradients(path="joint").explain(adapter, batch, "pred", K)
    own = IntegratedGradients(path="own").explain(adapter, batch, "pred", K)
    zero = IntegratedGradients(path="joint", baseline="zero").explain(adapter, batch, "pred", K)
    gap = completeness_gap(adapter, batch, own, "pred", K, baseline="mean")
    for i in range(len(batch.tickers)):
        s = lambda v: pd.Series(np.abs(v[i]).sum(0), index=names)
        checks.append({"date": batch.date, "own vs joint": agreement(s(own), s(joint))["spearman"],
                       "zero vs mean baseline": agreement(s(zero), s(joint))["spearman"],
                       "completeness gap / |change|": abs(gap[i]) / max(abs(own[i].sum() - gap[i]), 1e-12)})
pd.DataFrame(checks).describe().loc[["mean", "25%", "50%", "75%", "min"]]

,date,own vs joint,zero vs mean baseline,completeness gap / |change|
mean,2020-01-14 00:00:00,0.9991,0.8936,0.0296
25%,2019-12-31 00:00:00,0.9984,0.8779,0.0041
50%,2020-01-14 00:00:00,0.9992,0.8950,0.0097
75%,2020-01-28 00:00:00,0.9997,0.9122,0.0242
min,2019-12-31 00:00:00,0.9969,0.8123,0.0001


## Observations

MASTER, DJI, T = 20, L = 5, seed 42, test year 2020: 27 explained days, 737 stock-days, 4400 inputs per stock.

1. **The model uses every family, none dominates.** Net shares of IG (the families' total contributions, which match exact Shapley values, notebook 06): trend 21%, position 18%, momentum 15%, volume 15%, volatility 13%, candle 11%, price-volume 8%. The gross shares (sum of |attributions|) rank position first (27%) because its 45 features push in opposite directions within the same stock-day. The market family gets zero by construction (day-mean baseline); its effect goes through the gate (point 5).
2. **The top features come in groups of near-duplicates.** The 20 most important features (section 3) include pairs and triples of the same operator over different windows or of algebraically related operators: VSUMN5 and VSUMD5 (VSUMD = 2 VSUMP - 1), BETA20 and BETA30, RESI20 and RESI30, STD10 and STD60. The credit of an information is split among its duplicates (chapter 3), so the features have to be read by correlation group. By horizon, 38% of the attribution goes to long windows (21-60 days), 37% to medium, 25% to short ones.
3. **Recent days matter most, but the whole window is used.** The last input day receives 23% of the attribution, the last five days 52%, the oldest five days still 16%.
4. **Score and portfolio membership have the same explanation.** IG of the score and of the top-k margin agree almost perfectly (Spearman 0.9998 on features, identical rankings of families and lags): moving a stock's own inputs barely moves the boundary.
5. **The methods agree on what moves the score, not on what carries skill.** IG and Saliency rank the families alike (Spearman 0.96; 0.90 on features), and both agree with the permutation measure of the portfolio change (0.75). The permutation drop in RankIC ranks the families differently (0.64 with IG, 0.21 with the portfolio change): trend and position replace the most stocks of the top 5 when shuffled (53% and 47%) but carry no skill (drop in RankIC -0.007 and +0.004), while momentum, volume and volatility carry the little skill there is (+0.03 each).
6. **MASTER's own weights tell a different story from IG.** The replay of the forward pass reproduces the model exactly. The feature gate rescales the features by up to 2.1 depending on the market state (day-to-day standard deviation 0.15), up-weighting KSFT2, CNTD30, CORR60, CORR5 by up to 1.3 on average, but its weights barely relate to IG's feature importance (Spearman 0.32; 1 shared feature in the top 10): the gate amplifies features that the rest of the network then uses little. The temporal attention puts 71% of its weight on the last day and 99.7% on the last three, while IG gives the last three days only 46% of the attribution: attention is not attribution. The inter-stock attention is uniform (weight on oneself equal to 1/N).
7. **The IG settings are sound.** The joint path ranks each stock's features like the guide's own path (median Spearman 0.999); the zero baseline gives a somewhat different ranking (median 0.89), so the choice of the day-mean baseline matters and is reported; the own path's completeness gap is 1% of the change of the score (median), so 16 integration steps suffice.